# Gold Meta-Model FINAL - Production Model

**Purpose:** Train on ALL available data (no train/val/test split) for production deployment.

**Architecture:** XGBoost (attempt 7 exact hyperparameters) + Bootstrap Data Subsampling Ensemble (12 models)

**Based on:** Attempt 17 (validated: DA 60.04%, HCDA 64.13%, Sharpe 2.46)

**Key difference from attempt 17:**
- ALL data used for training (no holdout)
- OLS scaling factor fixed at 0.5 (from attempt 17 validation)
- No evaluation metrics (no test set to evaluate against)
- Outputs: model weights + full predictions with confidence scores

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import json
import os
import glob
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print(f"XGBoost version: {xgb.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Started: {datetime.now().isoformat()}")
print(f"Mode: FINAL PRODUCTION MODEL - Full Data Training")

In [ ]:
print("="*60)
print("LOADING DATA FROM KAGGLE DATASET")
print("="*60)

# Dataset path resolution
_PROBE_FILE_BF = 'base_features_raw.csv'
_glob_patterns_bf = [
    f'/kaggle/input/datasets/bigbigzabuton/gold-prediction-submodels/{_PROBE_FILE_BF}',
    f'/kaggle/input/gold-prediction-submodels/{_PROBE_FILE_BF}',
    f'/kaggle/input/datasets/*/gold-prediction-submodels/{_PROBE_FILE_BF}',
    f'/kaggle/input/*/{_PROBE_FILE_BF}',
]

DATASET_BASE_EARLY = None
for _pattern in _glob_patterns_bf:
    _matches = glob.glob(_pattern)
    if _matches:
        _candidate_base = os.path.dirname(_matches[0])
        try:
            pd.read_csv(_matches[0], nrows=1)
            DATASET_BASE_EARLY = _candidate_base
            print(f"Dataset found at: {DATASET_BASE_EARLY} (pattern: {_pattern})")
            break
        except Exception as _e:
            print(f"  Found at {_matches[0]} but read failed: {_e}")

if DATASET_BASE_EARLY is None:
    print("ERROR: base_features_raw.csv not found! Searched:")
    for _p in _glob_patterns_bf:
        print(f"  {_p} -> {glob.glob(_p)}")
    try:
        import subprocess as _sp
        _r = _sp.run(['find', '/kaggle/input', '-name', _PROBE_FILE_BF, '-type', 'f'],
                     capture_output=True, text=True, timeout=15)
        print(f"  find result: {_r.stdout.strip() or '(nothing found)'}")
    except Exception as _e:
        print(f"  find failed: {_e}")
    raise FileNotFoundError(
        "base_features_raw.csv not found in gold-prediction-submodels dataset."
    )

print("\nLoading base_features_raw.csv from dataset...")
bf = pd.read_csv(f'{DATASET_BASE_EARLY}/base_features_raw.csv')
bf['Date'] = pd.to_datetime(bf['Date']).dt.strftime('%Y-%m-%d')
bf = bf.set_index('Date')
print(f"  Loaded: {len(bf)} rows, {bf.index.min()} to {bf.index.max()}")
print(f"  Columns: {list(bf.columns)}")

base_features = bf[['gold_return_next', 'real_rate_real_rate', 'dxy_dxy', 'vix_vix',
                     'yield_curve_yield_spread',
                     'inflation_expectation_inflation_expectation']].copy()
base_features = base_features.ffill()
base_features = base_features.dropna(subset=['gold_return_next'])
print(f"  Base features: {len(base_features)} rows, {len(base_features.columns)} columns")
print("\nData loading complete")

In [ ]:
print("\nApplying transformations...")

final_df = base_features.copy()

final_df['real_rate_change'] = final_df['real_rate_real_rate'].diff()
final_df['dxy_change'] = final_df['dxy_dxy'].diff()
final_df['vix'] = final_df['vix_vix']
final_df['yield_spread_change'] = final_df['yield_curve_yield_spread'].diff()
final_df['inflation_exp_change'] = final_df['inflation_expectation_inflation_expectation'].diff()

final_df = final_df.drop(columns=['real_rate_real_rate', 'dxy_dxy', 'vix_vix',
                                    'yield_curve_yield_spread', 'inflation_expectation_inflation_expectation'])

print(f"  Base transformations applied")
print(f"  Columns so far: {list(final_df.columns)}")

In [ ]:
FEATURE_COLUMNS = [
    # Base features (5)
    'real_rate_change',
    'dxy_change',
    'vix',
    'yield_spread_change',
    'inflation_exp_change',
    # VIX submodel (3)
    'vix_regime_probability',
    'vix_mean_reversion_z',
    'vix_persistence',
    # Technical submodel (3)
    'tech_trend_regime_prob',
    'tech_mean_reversion_z',
    'tech_volatility_regime',
    # Cross-asset submodel (3)
    'xasset_regime_prob',
    'xasset_recession_signal',
    'xasset_divergence',
    # Yield curve submodel (2)
    'yc_spread_velocity_z',
    'yc_curvature_z',
    # ETF flow submodel (3)
    'etf_regime_prob',
    'etf_capital_intensity',
    'etf_pv_divergence',
    # Inflation expectation submodel (3)
    'ie_regime_prob',
    'ie_anchoring_z',
    'ie_gold_sensitivity_z',
    # Options market submodel (1)
    'options_risk_regime_prob',
    # Temporal context submodel (1)
    'temporal_context_score',
]

TARGET = 'gold_return_next'

assert len(FEATURE_COLUMNS) == 24, f"Expected 24 features, got {len(FEATURE_COLUMNS)}"
print(f"Features defined: {len(FEATURE_COLUMNS)} features")

In [ ]:
# Dataset path resolution for submodel outputs
import pandas as _pd_probe
DATASET_SLUG = 'gold-prediction-submodels'
DATASET_OWNER = 'bigbigzabuton'
_PROBE_FILE = 'vix.csv'

_glob_patterns = [
    f'/kaggle/input/datasets/{DATASET_OWNER}/{DATASET_SLUG}/{_PROBE_FILE}',
    f'/kaggle/input/{DATASET_SLUG}/{_PROBE_FILE}',
    f'/kaggle/input/datasets/*/{DATASET_SLUG}/{_PROBE_FILE}',
    f'/kaggle/input/*/{_PROBE_FILE}',
]

DATASET_BASE = None
for _pattern in _glob_patterns:
    _matches = glob.glob(_pattern)
    if _matches:
        _candidate_base = os.path.dirname(_matches[0])
        try:
            _pd_probe.read_csv(_matches[0], nrows=1)
            DATASET_BASE = _candidate_base
            print(f"Dataset found at: {DATASET_BASE} (pattern: {_pattern})")
            break
        except Exception as _e:
            print(f"  Found at {_matches[0]} but read failed: {_e}")

if DATASET_BASE is None:
    print("ERROR: Dataset not found via glob! Searched patterns:")
    for _p in _glob_patterns:
        print(f"  {_p} -> {glob.glob(_p)}")
    raise FileNotFoundError(
        f"Dataset '{DATASET_SLUG}' not found. "
        "Ensure dataset_sources includes 'bigbigzabuton/gold-prediction-submodels'."
    )

In [ ]:
print("\nLoading submodel outputs from Kaggle Dataset...")

submodel_files = {
    'vix': {
        'path': f'{DATASET_BASE}/vix.csv',
        'columns': ['vix_regime_probability', 'vix_mean_reversion_z', 'vix_persistence'],
        'date_col': 'date',
        'tz_aware': False,
        'rename': {},
    },
    'technical': {
        'path': f'{DATASET_BASE}/technical.csv',
        'columns': ['tech_trend_regime_prob', 'tech_mean_reversion_z', 'tech_volatility_regime'],
        'date_col': 'date',
        'tz_aware': True,
        'rename': {},
    },
    'cross_asset': {
        'path': f'{DATASET_BASE}/cross_asset.csv',
        'columns': ['xasset_regime_prob', 'xasset_recession_signal', 'xasset_divergence'],
        'date_col': 'Date',
        'tz_aware': False,
        'rename': {},
    },
    'yield_curve': {
        'path': f'{DATASET_BASE}/yield_curve.csv',
        'columns': ['yc_spread_velocity_z', 'yc_curvature_z'],
        'date_col': 'index',
        'tz_aware': False,
        'rename': {},
    },
    'etf_flow': {
        'path': f'{DATASET_BASE}/etf_flow.csv',
        'columns': ['etf_regime_prob', 'etf_capital_intensity', 'etf_pv_divergence'],
        'date_col': 'Date',
        'tz_aware': False,
        'rename': {},
    },
    'inflation_expectation': {
        'path': f'{DATASET_BASE}/inflation_expectation.csv',
        'columns': ['ie_regime_prob', 'ie_anchoring_z', 'ie_gold_sensitivity_z'],
        'date_col': 'Unnamed: 0',
        'tz_aware': False,
        'rename': {},
    },
    'options_market': {
        'path': f'{DATASET_BASE}/options_market.csv',
        'columns': ['options_risk_regime_prob'],
        'date_col': 'Date',
        'tz_aware': True,
        'rename': {'options_regime_smooth': 'options_risk_regime_prob'},
    },
    'temporal_context': {
        'path': f'{DATASET_BASE}/temporal_context.csv',
        'columns': ['temporal_context_score'],
        'date_col': 'date',
        'tz_aware': False,
        'rename': {},
    },
}

submodel_dfs = {}
for feature, spec in submodel_files.items():
    df = pd.read_csv(spec['path'])

    # Rename columns if needed (dataset column names may differ)
    if spec.get('rename'):
        df = df.rename(columns=spec['rename'])

    date_col = spec['date_col']
    if spec['tz_aware']:
        df['Date'] = pd.to_datetime(df[date_col], utc=True).dt.strftime('%Y-%m-%d')
    else:
        if date_col == 'index':
            df['Date'] = pd.to_datetime(df.iloc[:, 0]).dt.strftime('%Y-%m-%d')
        elif date_col == 'Unnamed: 0':
            df['Date'] = pd.to_datetime(df['Unnamed: 0']).dt.strftime('%Y-%m-%d')
        else:
            df['Date'] = pd.to_datetime(df[date_col]).dt.strftime('%Y-%m-%d')
    df = df[['Date'] + spec['columns']]
    df = df.set_index('Date')
    submodel_dfs[feature] = df
    print(f"  {feature}: {len(df)} rows")

print("\nMerging submodel outputs...")
for feature, df in submodel_dfs.items():
    final_df = final_df.join(df, how='left')
print(f"  Features after merge: {final_df.shape[1]} columns, {len(final_df)} rows")

print("\nApplying NaN imputation...")
nan_before = final_df.isna().sum().sum()
print(f"  NaN before imputation: {nan_before}")

regime_cols = ['vix_regime_probability', 'tech_trend_regime_prob',
               'xasset_regime_prob', 'etf_regime_prob', 'ie_regime_prob',
               'options_risk_regime_prob', 'temporal_context_score']
for col in regime_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0.5)

z_cols = ['vix_mean_reversion_z', 'tech_mean_reversion_z',
          'yc_spread_velocity_z', 'yc_curvature_z',
          'etf_capital_intensity', 'etf_pv_divergence',
          'ie_anchoring_z', 'ie_gold_sensitivity_z']
for col in z_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0.0)

div_cols = ['xasset_recession_signal', 'xasset_divergence']
for col in div_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0.0)

cont_cols = ['tech_volatility_regime', 'vix_persistence']
for col in cont_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(final_df[col].median())

final_df = final_df.dropna(subset=['gold_return_next', 'real_rate_change', 'dxy_change',
                                     'vix', 'yield_spread_change', 'inflation_exp_change'])

nan_after = final_df.isna().sum().sum()
print(f"  NaN after imputation: {nan_after}")
print(f"  Final dataset: {len(final_df)} rows")

assert all(col in final_df.columns for col in FEATURE_COLUMNS), "Missing features!"
assert TARGET in final_df.columns, "Target not found!"
print(f"\nAll {len(FEATURE_COLUMNS)} features present")
print(f"Dataset shape: {final_df.shape}")
print(f"Date range: {final_df.index.min()} to {final_df.index.max()}")

In [ ]:
# ============================================================
# FULL DATA - NO TRAIN/VAL/TEST SPLIT
# All data used for training the production model
# ============================================================
print("="*60)
print("PRODUCTION MODE: ALL DATA FOR TRAINING (NO SPLIT)")
print("="*60)

X_all = final_df[FEATURE_COLUMNS].values
y_all = final_df[TARGET].values
dates_all = final_df.index.tolist()

print(f"\nTraining data: {X_all.shape[0]} rows x {X_all.shape[1]} features")
print(f"Date range: {dates_all[0]} to {dates_all[-1]}")
print(f"Samples per feature: {X_all.shape[0] / X_all.shape[1]:.1f}:1")
print(f"\nTarget distribution:")
print(f"  Mean:     {y_all.mean():.4f}%")
print(f"  Std:      {y_all.std():.4f}%")
print(f"  Positive: {(y_all > 0).sum() / len(y_all) * 100:.1f}%")

In [ ]:
# ============================================================
# XGBoost Hyperparameters - Attempt 7 Exact Values (NO Optuna)
# ============================================================
print("="*60)
print("XGBOOST HYPERPARAMETERS (ATTEMPT 7 EXACT - NO HPO)")
print("="*60)

xgb_params = {
    "objective": "reg:squarederror",
    "max_depth": 2,
    "min_child_weight": 25,
    "subsample": 0.765,
    "colsample_bytree": 0.450,
    "reg_lambda": 2.049,
    "reg_alpha": 1.107,
    "learning_rate": 0.0215,
    "n_estimators": 621,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "verbosity": 0,
}

# OLS scaling factor fixed from attempt 17 validation
ALPHA_OLS_FIXED = 0.5

print("XGBoost parameters (from attempt 7):")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")
print(f"\nOLS scaling factor (fixed from attempt 17): {ALPHA_OLS_FIXED}")

In [ ]:
# ============================================================
# Bootstrap Data Subsampling Ensemble Training
# Each model trains on a different 80% bootstrap sample of ALL data
# ============================================================
print("="*60)
print("BOOTSTRAP DATA SUBSAMPLING ENSEMBLE TRAINING (FULL DATA)")
print("="*60)

N_ENSEMBLE = 12
BOOTSTRAP_FRAC = 0.80

rng = np.random.RandomState(42)
ensemble_models = []

n_bootstrap = int(BOOTSTRAP_FRAC * len(X_all))
print(f"\nEnsemble config:")
print(f"  Number of models: {N_ENSEMBLE}")
print(f"  Bootstrap fraction: {BOOTSTRAP_FRAC:.0%}")
print(f"  Bootstrap sample size: {n_bootstrap} / {len(X_all)} total rows")
print(f"  Mode: PRODUCTION (all data, no holdout)")
print()

for i in range(N_ENSEMBLE):
    seed = 42 + i
    bootstrap_idx = rng.choice(len(X_all), size=n_bootstrap, replace=True)
    X_boot = X_all[bootstrap_idx]
    y_boot = y_all[bootstrap_idx]

    model_params = xgb_params.copy()
    model_params['random_state'] = seed

    model = xgb.XGBRegressor(**model_params)
    model.fit(X_boot, y_boot, verbose=False)
    ensemble_models.append(model)
    unique_rows = len(np.unique(bootstrap_idx))
    print(f"  Model {i+1:2d}/{N_ENSEMBLE}: seed={seed}, unique_rows={unique_rows}, bootstrap_size={n_bootstrap}")

print(f"\nEnsemble training complete: {len(ensemble_models)} models")

In [ ]:
# ============================================================
# Ensemble Predictions
# ============================================================
print("="*60)
print("ENSEMBLE PREDICTIONS")
print("="*60)

ensemble_preds = np.array([m.predict(X_all) for m in ensemble_models])

pred_raw = ensemble_preds.mean(axis=0)
bootstrap_std = ensemble_preds.std(axis=0)

# Apply fixed OLS scaling
pred_scaled = pred_raw * ALPHA_OLS_FIXED

print(f"\nRaw ensemble predictions:")
print(f"  Mean: {pred_raw.mean():.4f}, Std: {pred_raw.std():.4f}")
print(f"  Min:  {pred_raw.min():.4f}, Max: {pred_raw.max():.4f}")
print(f"  Positive: {(pred_raw > 0).sum() / len(pred_raw) * 100:.1f}%")

print(f"\nScaled predictions (alpha={ALPHA_OLS_FIXED}):")
print(f"  Mean: {pred_scaled.mean():.4f}, Std: {pred_scaled.std():.4f}")

print(f"\nBootstrap diversity:")
print(f"  Std range: [{bootstrap_std.min():.4f}, {bootstrap_std.max():.4f}]")
print(f"  Std mean:  {bootstrap_std.mean():.4f}")

In [ ]:
# ============================================================
# In-Sample Metrics (reference only, not for model selection)
# ============================================================
print("="*60)
print("IN-SAMPLE METRICS (Reference Only)")
print("="*60)

def compute_direction_accuracy(y_true, y_pred):
    mask = (y_true != 0) & (y_pred != 0)
    if mask.sum() == 0:
        return 0.0
    return (np.sign(y_pred[mask]) == np.sign(y_true[mask])).mean()

def compute_mae(y_true, y_pred):
    return np.abs(y_pred - y_true).mean()

def compute_sharpe_trade_cost(y_true, y_pred, cost_bps=5.0):
    positions = np.sign(y_pred)
    strategy_returns = positions * y_true / 100.0
    position_changes = np.abs(np.diff(positions, prepend=0))
    trade_costs = position_changes * (cost_bps / 10000.0)
    net_returns = strategy_returns - trade_costs
    if len(net_returns) < 2 or net_returns.std() == 0:
        return 0.0
    return (net_returns.mean() / net_returns.std()) * np.sqrt(252)

da = compute_direction_accuracy(y_all, pred_raw)
mae_raw = compute_mae(y_all, pred_raw)
mae_scaled = compute_mae(y_all, pred_scaled)
sharpe = compute_sharpe_trade_cost(y_all, pred_raw)

print(f"\nIn-sample (ALL {len(y_all)} rows):")
print(f"  DA:     {da*100:.2f}%")
print(f"  MAE:    {min(mae_raw, mae_scaled):.4f}% (raw: {mae_raw:.4f}%, scaled: {mae_scaled:.4f}%)")
print(f"  Sharpe: {sharpe:.2f}")
print(f"\nNote: These are in-sample metrics. Out-of-sample performance")
print(f"was validated in attempt 17 (DA=60.04%, Sharpe=2.46 on test set).")

In [ ]:
# ============================================================
# Feature Importance (average across ensemble)
# ============================================================
print("="*60)
print("FEATURE IMPORTANCE")
print("="*60)

importances_list = [m.feature_importances_ for m in ensemble_models]
avg_importance = np.mean(importances_list, axis=0)
std_importance = np.std(importances_list, axis=0)

feature_importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance_mean': avg_importance,
    'importance_std': std_importance,
}).sort_values('importance_mean', ascending=False)

total_imp = feature_importance_df['importance_mean'].sum()
feature_importance_df['importance_pct'] = feature_importance_df['importance_mean'] / max(total_imp, 1e-10) * 100
feature_importance_df = feature_importance_df.reset_index(drop=True)

print("\nAll features (ranked by importance):")
for i, row in feature_importance_df.iterrows():
    print(f"  {i+1:2d}. {row['feature']}: {row['importance_pct']:.2f}%")

In [ ]:
# ============================================================
# Save Results
# ============================================================
print("="*60)
print("SAVING RESULTS")
print("="*60)

# --- Confidence scores ---
# Bootstrap std: bottom 20% = high confidence
hc_std_threshold = np.percentile(bootstrap_std, 20)
hc_mask_std = (bootstrap_std <= hc_std_threshold).astype(int)

# |prediction|: top 20% = high confidence
hc_abs_threshold = np.percentile(np.abs(pred_raw), 80)
hc_mask_abs = (np.abs(pred_raw) >= hc_abs_threshold).astype(int)

# --- predictions.csv (all data) ---
predictions_df = pd.DataFrame({
    'date': dates_all,
    'actual': y_all,
    'prediction': pred_raw,
    'prediction_scaled': pred_scaled,
    'bootstrap_std': bootstrap_std,
    'high_confidence_std': hc_mask_std,
    'high_confidence_abs': hc_mask_abs,
    'direction_correct': (np.sign(pred_raw) == np.sign(y_all)).astype(int),
})
predictions_df.to_csv('predictions.csv', index=False)
print("Saved predictions.csv")

# --- Save ensemble models ---
for i, model in enumerate(ensemble_models):
    model.save_model(f'ensemble_model_{i}.json')
print(f"Saved {N_ENSEMBLE} ensemble model files (ensemble_model_*.json)")

# --- Save model config for inference ---
inference_config = {
    'feature_columns': FEATURE_COLUMNS,
    'n_ensemble': N_ENSEMBLE,
    'alpha_ols': ALPHA_OLS_FIXED,
    'xgb_params': xgb_params,
    'bootstrap_fraction': BOOTSTRAP_FRAC,
    'hc_std_threshold': float(hc_std_threshold),
    'hc_abs_threshold': float(hc_abs_threshold),
    'training_date_range': [dates_all[0], dates_all[-1]],
    'training_samples': len(dates_all),
}
with open('inference_config.json', 'w') as f:
    json.dump(inference_config, f, indent=2)
print("Saved inference_config.json")

# --- Save feature importance ---
feature_importance_df.to_csv('feature_importance.csv', index=False)
print("Saved feature_importance.csv")

# --- training_result.json ---
fi_top10 = feature_importance_df.head(10)[['feature', 'importance_pct']].to_dict('records')

training_result = {
    'feature': 'meta_model',
    'attempt': 'FINAL',
    'timestamp': datetime.now().isoformat(),
    'architecture': 'XGBoost (attempt 7 params) + Bootstrap Data Subsampling Ensemble (12 models)',
    'mode': 'PRODUCTION - Full Data Training (no train/val/test split)',
    'based_on': 'attempt 17 (validated: DA 60.04%, HCDA 64.13%, Sharpe 2.46)',

    'model_config': {
        'algorithm': 'XGBoost',
        'n_features': len(FEATURE_COLUMNS),
        'total_samples': len(X_all),
        'samples_per_feature_ratio': len(X_all) / len(FEATURE_COLUMNS),
        'xgb_params': xgb_params,
        'n_ensemble': N_ENSEMBLE,
        'bootstrap_fraction': BOOTSTRAP_FRAC,
        'bootstrap_sample_size': n_bootstrap,
        'alpha_ols_fixed': ALPHA_OLS_FIXED,
    },

    'training_data': {
        'date_range': [dates_all[0], dates_all[-1]],
        'total_rows': len(dates_all),
        'split': 'NONE - all data used for training',
    },

    'in_sample_metrics': {
        'direction_accuracy': float(da),
        'mae_raw': float(mae_raw),
        'mae_scaled': float(mae_scaled),
        'sharpe_ratio': float(sharpe),
        'note': 'In-sample only. Out-of-sample validated in attempt 17.',
    },

    'validated_oos_metrics_attempt17': {
        'direction_accuracy': 0.6004,
        'high_confidence_da': 0.6413,
        'sharpe_ratio': 2.46,
        'targets_passed': '3/4',
    },

    'bootstrap_diversity': {
        'std_range': [float(bootstrap_std.min()), float(bootstrap_std.max())],
        'std_mean': float(bootstrap_std.mean()),
    },

    'confidence_thresholds': {
        'hc_std_threshold': float(hc_std_threshold),
        'hc_abs_threshold': float(hc_abs_threshold),
    },

    'feature_importance': {
        'method': 'xgboost_gain_averaged_over_ensemble',
        'top_10': fi_top10,
    },

    'output_files': [
        'predictions.csv',
        'inference_config.json',
        'feature_importance.csv',
        'training_result.json',
    ] + [f'ensemble_model_{i}.json' for i in range(N_ENSEMBLE)],
}

with open('training_result.json', 'w') as f:
    json.dump(training_result, f, indent=2, default=str)
print("Saved training_result.json")

print(f"\n{'='*60}")
print("PRODUCTION MODEL TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Finished: {datetime.now().isoformat()}")
print(f"\nFinal Summary:")
print(f"  Architecture: XGBoost + Bootstrap Ensemble ({N_ENSEMBLE} models)")
print(f"  Training data: {len(X_all)} rows ({dates_all[0]} to {dates_all[-1]})")
print(f"  Features: {len(FEATURE_COLUMNS)}")
print(f"  OLS alpha: {ALPHA_OLS_FIXED}")
print(f"  In-sample DA: {da*100:.2f}%")
print(f"  In-sample Sharpe: {sharpe:.2f}")
print(f"  Validated OOS (attempt 17): DA=60.04%, Sharpe=2.46")
print(f"\nOutput files: {len(training_result['output_files'])} files saved")
print(f"\nThis model is ready for production inference.")